# Data Collection

This notebook pulls price and fundamental data for every company in the curated universe (`data/raw/universe.csv`), using yfinance. Each pull is checked for completeness and logged to `data/raw/collection_log.csv`, so failures and gaps can be reviewed and addressed afterward.

In [ ]:
import sys
sys.path.append("..")

import src.data_collection as dc
from src import config

import pandas as pd

## Load Universe

Read in the curated universe list. Also check for duplicate tickers before proceeding, to catch any accidental double-entries in the manually built list.

In [3]:
universe = pd.read_csv("../data/raw/universe.csv")
universe["Ticker"].duplicated().sum()
print(universe)

                       Company  Ticker Country   Band  \
0                     RELX plc   REL.L      UK   Mega   
1                 Experian plc  EXPN.L      UK  Large   
2        Auto Trader Group plc  AUTO.L      UK    Mid   
3                Rightmove plc   RMV.L      UK    Mid   
4                       RM plc    RM.L      UK  Small   
..                         ...     ...     ...    ...   
116                Cintas Corp    CTAS      US  Large   
117    Jack Henry & Associates    JKHY      US    Mid   
118                Rollins Inc     ROL      US  Large   
119              UniFirst Corp     UNF      US    Mid   
120  Healthcare Services Group    HCSG      US  Small   

                        Sector  \
0         Information Services   
1         Information Services   
2         Information Services   
3         Information Services   
4         Information Services   
..                         ...   
116  Mission-Critical Services   
117  Mission-Critical Services   
118  Miss

## Configuration

Date range, required fields, and save directories are defined in `src/config.py` (shared across notebooks so all collection and review steps use consistent settings). See that file for details and rationale on the current values.

## Collection Loop

For each company in the universe, pull all five data types in a single pass (prices, balance sheet, cash flow, income statement, metadata). Each pull is:

1. Wrapped in a try/except, so one company's failure doesn't stop the loop.
2. Checked for missing data (empty results, missing required fields, fully empty periods).
3. Cleaned (fully-empty periods dropped) and saved to `data/raw/` — one CSV per ticker for prices, one folder per ticker for fundamentals statements, and one combined `metadata.csv` for all companies.
4. Logged to `collection_log.csv`, recording status, row/period counts, date range, and any missing data — this log is reviewed in the next notebook (`02_review_collection_log.ipynb`) to identify and fix data issues (wrong tickers, delisted companies, insufficient history).

In [ ]:
log_entries = []
metadata_entries = []

for ticker in universe["Ticker"]:
    ticker_logs, metadata_entry = dc.collect_all_data(
        ticker, 
        config.START_DATE, 
        config.END_DATE,
        config.BALANCE_SHEET_REQUIRED_FIELDS,
        config.CASH_FLOW_REQUIRED_FIELDS,
        config.INCOME_STATEMENT_REQUIRED_FIELDS,
        config.METADATA_REQUIRED_FIELDS,
        config.PRICES_SAVE_DIR,
        config.FUNDAMENTALS_SAVE_DIR
    )

    if metadata_entry is not None:
        metadata_entries.append(metadata_entry)
    log_entries.extend(ticker_logs)

metadata_df = pd.DataFrame(metadata_entries)
metadata_df.to_csv("../data/raw/metadata.csv", index = False)

collection_log = pd.DataFrame(log_entries)
collection_log.to_csv("../data/raw/collection_log.csv", index = False)

$DNB: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DNB"}}}
$SXS.L: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SXS.L"}}}
$TET.L: possibly delisted; no price data found  (1d 2022-01-01 -> 2025-12-31) (Yahoo error = "Data doesn't exist for startDate = 1640995200, endDate = 1767139200")
$RWI.L: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: RWI.L"}}}
$MEG: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MEG"}}}
$APH.L: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found f

## Review Collection Log

After running the collection loop, check `collection_log.csv` to deal with any failed or partial pulls and keep re-running this notebook until the universe is stable. 

## Re-run Collection for Fixed Tickers

Two tickers were corrected in `universe.csv` following the failure review: `MEG` → `ONT` (Montrose Environmental renamed to Onterris Inc.) and `FI` → `FISV` (incorrect ticker symbol for Fiserv).

Rather than re-running the full collection loop, `collect_all_data` is called directly for just these two corrected tickers. The old entries for `MEG` and `FI` are removed from `collection_log.csv` and `metadata.csv`, and the new results are appended in their place — keeping both files accurate and up to date without duplicating a full re-pull of the entire universe.

## Clean Up Log for Removed Companies

The 6 companies confirmed as delisted/acquired (see markdown above) have been removed from `universe.csv`. Their corresponding failed entries are also removed from `collection_log.csv`, since they are no longer part of the universe and don't need further investigation.

In [ ]:
fixed_tickers = ["ONT", "FISV"]
log_entries = []
metadata_entries = []

for ticker in fixed_tickers:
    ticker_logs, metadata_entry = dc.collect_all_data(
        ticker, 
        config.START_DATE, 
        config.END_DATE,
        config.BALANCE_SHEET_REQUIRED_FIELDS,
        config.CASH_FLOW_REQUIRED_FIELDS,
        config.INCOME_STATEMENT_REQUIRED_FIELDS,
        config.METADATA_REQUIRED_FIELDS,
        config.PRICES_SAVE_DIR,
        config.FUNDAMENTALS_SAVE_DIR
    )

    if metadata_entry is not None:
        metadata_entries.append(metadata_entry)
    log_entries.extend(ticker_logs)

# Update collection_log.csv

removed_tickers = ["DNB", "SXS.L", "TET.L", "RWI.L", "APH.L", "ALPH.L", "MEG", "FI"]

existing_log = pd.read_csv("../data/raw/collection_log.csv")
new_log = pd.DataFrame(log_entries)

existing_log = existing_log[~existing_log["ticker"].isin(removed_tickers)] # remove existing logs of MEG and FI

updated_log = pd.concat([existing_log, new_log], ignore_index = True)
updated_log.to_csv("../data/raw/collection_log.csv", index = False)

# Update metadata.csv

existing_metadata = pd.read_csv("../data/raw/metadata.csv")
new_metadata = pd.DataFrame(metadata_entries)

existing_metadata = existing_metadata[~existing_metadata["ticker"].isin(["MEG", "FI"])] # remove existing metadatas of MEG and FI

updated_metadata = pd.concat([existing_metadata, new_metadata], ignore_index = True)
updated_metadata.to_csv("../data/raw/metadata.csv", index = False)
    